# Vectora: Linear Algebra and Numerical Computing Walkthrough

A reproducible companion to the Vectora Scientific Studio. This notebook uses NumPy for reference computations and makes each experiment explicit. Run cells in order.

## 1. Matrices, dimensions, and matrix multiplication

For A of shape m×n and B of shape n×p, AB is defined and has shape m×p. The inner dimensions must match.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[2., 1., 1.], [1., 3., 2.], [1., 0., 2.]])
B = np.array([[1., 2., 0.], [0., 1., 1.], [2., 0., 1.]])
print('A shape:', A.shape)
print('B shape:', B.shape)
print('A @ B =\n', A @ B)
print('transpose(A) =\n', A.T)
print('det(A):', np.linalg.det(A))
print('rank(A):', np.linalg.matrix_rank(A))

## 2. Solve a system and inspect its residual

A numerical answer should be checked by substituting it back into the original system. The residual is r = Ax − b.

In [ ]:
b = np.array([4., 7., 3.])
x = np.linalg.solve(A, b)
residual = A @ x - b
print('solution:', x)
print('residual:', residual)
print('residual norm:', np.linalg.norm(residual))

## 3. LU and QR factorization

LU with pivoting factors PA = LU. QR factors A = QR, where Q has orthonormal columns and R is upper triangular (for a full-rank square matrix).

In [ ]:
from scipy.linalg import lu, qr

P, L, U = lu(A)
Q, R = qr(A)
print('LU reconstruction error:', np.linalg.norm(P @ A - L @ U))
print('QR reconstruction error:', np.linalg.norm(A - Q @ R))
print('Q.T @ Q =\n', Q.T @ Q)

## 4. SVD, pseudoinverse, and conditioning

The SVD A = UΣVᵀ exposes singular directions. Small singular values indicate directions that are strongly attenuated and can make inversion sensitive to noise.

In [ ]:
U, s, Vt = np.linalg.svd(A, full_matrices=False)
A_reconstructed = U @ np.diag(s) @ Vt
A_pinv = np.linalg.pinv(A)
print('singular values:', s)
print('SVD reconstruction error:', np.linalg.norm(A - A_reconstructed))
print('condition number:', np.linalg.cond(A))
print('pseudoinverse identity error:', np.linalg.norm(A @ A_pinv @ A - A))

## 5. Least-squares regression

Least squares chooses coefficients that minimize the squared residual norm. The coefficient of determination R² describes the fit on this dataset, not causality.

In [ ]:
x_obs = np.array([0., 1., 2., 3., 4., 5.])
y_obs = np.array([1.1, 2.0, 2.4, 3.8, 4.1, 5.2])
design = np.column_stack([np.ones_like(x_obs), x_obs])
coef, *_ = np.linalg.lstsq(design, y_obs, rcond=None)
fitted = design @ coef
residual = y_obs - fitted
ss_res = residual @ residual
ss_tot = ((y_obs - y_obs.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print('intercept, slope:', coef)
print('R²:', r2)
print('residuals:', residual)

## 6. PCA on a two-feature dataset

Center each feature, compute its covariance matrix, then inspect eigenvectors in descending eigenvalue order. Standardization may be appropriate when feature units differ.

In [ ]:
data = np.array([[1, 1.2], [1.5, 1.7], [2, 1.8], [2.5, 2.7], [3, 2.8], [3.5, 3.4], [4, 4.1], [4.5, 4.2]], dtype=float)
centered = data - data.mean(axis=0)
cov = np.cov(centered, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
scores = centered @ eigenvectors
explained_ratio = eigenvalues / eigenvalues.sum()
print('covariance:\n', cov)
print('principal directions:\n', eigenvectors)
print('explained variance ratio:', explained_ratio)
print('first two projected rows:\n', scores[:2])

## 7. Geometry and homogeneous transforms

A 3×3 homogeneous transform combines 2D rotation and translation. Composition order matters because matrix multiplication is not generally commutative.

In [ ]:
theta = np.deg2rad(35)
T_world_local = np.array([[np.cos(theta), -np.sin(theta), 1.5], [np.sin(theta), np.cos(theta), -0.5], [0., 0., 1.]])
point_local = np.array([2., 1., 1.])
point_world = T_world_local @ point_local
point_recovered = np.linalg.inv(T_world_local) @ point_world
print('T =\n', T_world_local)
print('world point:', point_world[:2])
print('recovered local point:', point_recovered[:2])

## Interpretation checklist
- Verify dimensions before multiplying matrices.
- Inspect rank and singular values before trusting an inverse.
- Report residuals when solving systems or fitting models.
- State the tolerance and floating-point assumptions behind numerical classifications.
- Treat plots and fitted metrics as evidence about the chosen sample, not universal guarantees.